In [120]:
from datasets import load_dataset

ds = load_dataset("parquet",
                    data_files={'train': 'train.parquet',
                                "validation" : "validation.parquet",
                                'test': 'test.parquet'}
                    )

In [121]:
text_split = []

for row in ds["train"]["sentence"]:
    for w in row.split():
        text_split.append(w)

text_split = set(text_split)

In [122]:
s_i = {s:i+1 for i, s in enumerate(text_split)}
s_i["<n>"] = 0

i_s = {i:s for s, i in s_i.items()}

In [123]:
import torch

# building dataset

block_size = 8

def build_dataset(sents):


    X, Y = [], []
    for s in sents:
        #print(s)
        context = [0] * block_size
        s[-1] = "<n>"

        for w in s:

            ix = s_i[w]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [124]:
sents_split = [s.split() for s in ds["train"]["sentence"]]

Xtr, ytr = build_dataset(sents_split)

In [129]:
len([s.split() for s in ds["train"]["sentence"]]) + len([s.split() for s in ds["test"]["sentence"]]) + len([s.split() for s in ds["validation"]["sentence"]])

49199

In [9]:
dstr = torch.utils.data.TensorDataset(Xtr, ytr)

trloader = torch.utils.data.DataLoader(
    dataset=dstr,
    batch_size=500,
    shuffle=True,

)

In [10]:
# from torch_xla.core.xla_model import xm

""" class Reshape(torch.nn.Module):
    def __call__(self, x):
        # print(x.shape[0] / 2)
        if x.dim() == 2:
            return x.view((int(x.shape[0] / 2), -1))
        else:
            return torch.reshape(x, (int(x.shape[0]), int(x.shape[1] / 8), -1))

class Squeeze(torch.nn.Module):
    def __call__(self, x):
        return  """

class NN(torch.nn.Module):
    def __init__(self, vocab_size, emb_dim, n_hidden):
        super().__init__()

        self.vocab_size = vocab_size
        """
        self.layers = [
            torch.nn.Embedding(vocab_size, emb_dim),
            torch.nn.Linear(emb_dim, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Squeeze(), torch.nn.Linear(n_hidden, vocab_size)
        ] """

        self.embedding = torch.nn.Embedding(vocab_size, emb_dim)
        self.rnn = torch.nn.RNN(input_size=emb_dim, hidden_size=n_hidden, num_layers=4)
        # self.reshape = Reshape()
        # self.squeeze = Squeeze()
        self.l1 = torch.nn.Linear(n_hidden * 8, vocab_size)

        self.out = 0.0

        self.parameters_ = [p for p in self.embedding.parameters()] + [p for p in self.rnn.parameters()] + [p for p in self.l1.parameters()]

    """ def parameters(self):
        params = []

        for layer in self.layers:
            for p in layer.parameters:
                params.append(p)

        return params """

    def __call__(self, x, hn):
        x_ = x.to(device)

        x_ = self.embedding(x)
        # print(x_.shape)

        x_, hn = self.rnn(x_, hn)
        # print(x_.shape)

        single = True if x_.dim() == 2 else False

        x_ = x_.view((int(x_.shape[0] / 2), -1)) if single else torch.reshape(x_, (int(x_.shape[0]), int(x_.shape[1] / 8), -1))
        # print(x_.shape)

        x_ = torch.squeeze(x_)

        x_ = self.l1(x_)
        # print(x_.shape)

        self.out = x_
        return x_, hn

    def fit(self, max_iter, loader, lr):
        g = torch.Generator().manual_seed(2147483647)
        optimizer = torch.optim.AdamW(self.parameters_, lr=lr)

        lossi = []
        wattsi = []

        for p in self.parameters_:
            p.retain_grad()

        hn = None

        for step in range(max_iter):
            Xb, yb = next(iter(loader))

            Xb = Xb.to(device)
            yb = yb.to(device)

            if hn is not None:
                hn = hn.detach()

            logits, hn = self.__call__(Xb, hn)

            loss = torch.nn.functional.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            lossi.append(loss.item())

            print(f"\r{step} / {max_iter}: {loss:.6f}", end="", flush=True)

        return lossi


In [11]:
embedding_dim = 32

n_hidden = 100
vocab_size = len(s_i)

net = NN(vocab_size=vocab_size, emb_dim=embedding_dim, n_hidden=n_hidden)


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device.type)

cuda


In [ ]:
rnn = torch.nn.RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)

In [14]:
# net(Xtr[1], None)
net(Xtr[10].clone().expand(1, 8).to(device), None)

(tensor([ 0.0220, -0.0131, -0.0301,  ..., -0.0139,  0.1874, -0.0380],
        device='cuda:0', grad_fn=<ViewBackward0>),
 tensor([[[-3.8429e-01,  9.2439e-02, -8.3620e-02,  ..., -3.6665e-01,
           -7.0774e-01,  5.2597e-02],
          [ 2.4546e-01, -4.6432e-02, -2.8452e-01,  ..., -1.5082e-01,
           -2.7575e-01, -2.6948e-02],
          [ 4.3623e-01,  2.9406e-01, -6.3795e-01,  ...,  7.8916e-02,
            1.2571e-01,  7.1238e-02],
          ...,
          [ 1.0799e-01, -2.2080e-01,  1.9732e-01,  ...,  1.4619e-01,
            2.0262e-01,  3.0171e-02],
          [ 2.6543e-01, -1.2534e-01,  2.2307e-01,  ..., -9.5026e-02,
           -3.4496e-01, -1.0212e-01],
          [-4.4171e-01, -4.2566e-01,  1.0391e-01,  ..., -7.9826e-02,
           -1.1853e-02, -5.6208e-01]],
 
         [[ 4.2668e-01, -1.4454e-01, -9.9042e-02,  ...,  2.3614e-02,
            1.8889e-01,  1.1607e-01],
          [ 2.4633e-01, -3.0061e-01,  4.6238e-02,  ..., -6.1537e-03,
            1.7257e-01, -2.6839e-02],
     

In [15]:
lossi = net.fit(max_iter=10000,  loader=trloader, lr=(2.5 * 1e-3));

9999 / 10000: 4.189186

In [16]:
Xval, yval = build_dataset(s.split() for s in ds["validation"]["sentence"])
dsval = torch.utils.data.TensorDataset(Xval, yval)

valloader = torch.utils.data.DataLoader(dsval, batch_size=100)

In [17]:
def eval_model(model, loader, hn=None):
    total_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, h1 = model(x, hn)
            loss = torch.nn.functional.cross_entropy(logits, y)

            total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

In [18]:
print("Training set loss:", eval_model(net, trloader))
print("Validation set loss:", eval_model(net, valloader))

Training set loss: 4.046892409598809
Validation set loss: 5.910664436576345


In [19]:
g2 = torch.Generator(device).manual_seed(2147483647)
hni = None

for _ in range(5):
    out = []
    context = [0] * block_size


    while True:
        context = torch.tensor(context).to(device).expand(1, 8)
        probs, hni = net(context, hn=hni)
        logits = torch.nn.functional.softmax(probs, dim=0)

        ix = torch.multinomial(logits, num_samples=1, generator=g2).item()

        context = list(context)[1:] + [ix]

        if ix == 0:
            break

        out.append(ix)

        # if len(out) > 5:
        #     break
        # print(i_s[ix])


    print("".join(i_s[i] + " " for i in out))


iras and employees to 
notification yield N N N N N N N N N N N closing futures trading after trading 
american & and rises at N N N N N with its leasing 's unit group boston firm no fiscal nine N N N N days N N N yield N N N N N 
the league and a case the buy-out fell N 
texaco annual rate gain by <unk> <unk> <unk> to the crowd and some traders tell futures trading stock closed at par via international inc. offering fell N abc expects cuts to distribute N to first-time still as the limit traders 


In [74]:
# from torch_xla.core.xla_model import xm

class Reshape(torch.nn.Module):
    def __call__(self, x):
        # print(x.shape[0] / 2)
        if x.dim() == 2:
            return x.view((int(x.shape[0] / 2), -1))
        else:
            return torch.reshape(x, (int(x.shape[0]), int(x.shape[1] / 2), -1))

class Squeeze(torch.nn.Module):
    def __call__(self, x):
        return torch.squeeze(x)

class NN(torch.nn.Module):
    def __init__(self, vocab_size, emb_dim, n_hidden):
        super().__init__()

        self.vocab_size = vocab_size
        """
        self.layers = [
            torch.nn.Embedding(vocab_size, emb_dim),
            torch.nn.Linear(emb_dim, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Squeeze(), torch.nn.Linear(n_hidden, vocab_size)
        ] """

        self.embedding = torch.nn.Embedding(vocab_size, emb_dim)

        self.rnn1 = torch.nn.RNN(input_size=emb_dim, hidden_size=n_hidden, num_layers=1)
        self.ln1 = torch.nn.LayerNorm(n_hidden)

        self.rsh1 = Reshape()

        self.rnn2 = torch.nn.RNN(input_size=n_hidden * 2, hidden_size=n_hidden, num_layers=1)
        self.ln2 = torch.nn.LayerNorm(n_hidden)

        self.rsh2 = Reshape()

        self.rnn3 = torch.nn.RNN(input_size=n_hidden * 2, hidden_size=n_hidden, num_layers=1)
        self.ln3 = torch.nn.LayerNorm(n_hidden)

        self.rsh3 = Reshape()

        self.rnn4 = torch.nn.RNN(input_size=n_hidden * 2, hidden_size=n_hidden, num_layers=1)
        self.ln4 = torch.nn.LayerNorm(n_hidden)

        self.squeeze = Squeeze()

        self.l1 = torch.nn.Linear(n_hidden, vocab_size)

        self.out = 0.0

        layers = [self.embedding, self.rnn1, self.ln1, self.rnn2, self.ln2, self.rnn3, self.ln3, self.rnn4, self.ln4, self.l1]

        self.parameters_ = []

        for layer in layers:
            self.parameters_ += [p for p in layer.parameters()]

    """ def parameters(self):
        params = []

        for layer in self.layers:
            for p in layer.parameters:
                params.append(p)

        return params """

    def __call__(self, x, hn):
        x_ = x.to(device)

        x_ = self.embedding(x_)
        # print(x_.shape)

        x_, hn = self.rnn1(x_)
        # print(x_.shape)
        x_ = self.ln1(x_)
        # print(x_.shape)
        x_ = self.rsh1(x_)
        # print(x_.shape)

        x_, hn = self.rnn2(x_)
        # print(x_.shape)
        x_ = self.ln2(x_)
        # print(x_.shape)
        x_ = self.rsh2(x_)
        # print(x_.shape)

        x_, hn = self.rnn3(x_)
        # print(x_.shape)
        x_ = self.ln3(x_)
        # print(x_.shape)
        x_ = self.rsh3(x_)
        # print(x_.shape)

        x_, hn = self.rnn4(x_)
        # print(x_.shape)
        x_ = self.ln4(x_)
        # print(x_.shape)

        x_ = self.squeeze(x_)
        # print(x_.shape)

        x_ = self.l1(x_)
        # print(x_.shape)

        self.out = x_
        # print(x_.shape)
        return x_, hn

    def fit(self, max_iter, loader, lr):
        g = torch.Generator().manual_seed(2147483647)
        optimizer = torch.optim.AdamW(self.parameters_, lr=lr)

        lossi = []
        wattsi = []

        for p in self.parameters_:
            p.retain_grad()

        hn = None

        for step in range(max_iter):
            Xb, yb = next(iter(loader))

            Xb = Xb.to(device)
            yb = yb.to(device)

            if hn is not None:
                hn = hn.detach()

            logits, hn = self.__call__(Xb, hn)

            loss = torch.nn.functional.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            lossi.append(loss.item())

            print(f"\r{step} / {max_iter}: {loss:.6f}", end="", flush=True)

        return lossi

In [75]:
embedding_dim = 32

n_hidden = 100
vocab_size = len(s_i)

net = NN(vocab_size=vocab_size, emb_dim=embedding_dim, n_hidden=n_hidden)


In [76]:
net.to(device)

NN(
  (embedding): Embedding(10000, 32)
  (rnn1): RNN(32, 100)
  (ln1): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (rsh1): Reshape()
  (rnn2): RNN(200, 100)
  (ln2): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (rsh2): Reshape()
  (rnn3): RNN(200, 100)
  (ln3): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (rsh3): Reshape()
  (rnn4): RNN(200, 100)
  (ln4): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (squeeze): Squeeze()
  (l1): Linear(in_features=100, out_features=10000, bias=True)
)

In [77]:
# net(Xtr[1], None)
net(Xtr[10].clone().expand(1, 8).to(device), None)

(tensor([0.0554, 0.4688, 0.3622,  ..., 0.9576, 0.4517, 0.1824], device='cuda:0',
        grad_fn=<ViewBackward0>),
 tensor([[[-0.6835, -0.8457, -0.6269,  0.4233, -0.3735,  0.7724,  0.6660,
            0.1287,  0.0667,  0.0560, -0.0564, -0.6682,  0.1038,  0.1190,
           -0.2451,  0.4919,  0.8704, -0.7003,  0.7739, -0.6943,  0.8100,
            0.5572, -0.6502, -0.0864, -0.1383,  0.7928,  0.8821,  0.0916,
            0.6253,  0.8099, -0.8231, -0.2445, -0.0666, -0.4957,  0.4841,
           -0.3672,  0.2817,  0.7815,  0.0799,  0.7132,  0.5445,  0.3369,
            0.6939,  0.1956, -0.3792,  0.7471, -0.8285, -0.5499,  0.7208,
            0.5740,  0.5961,  0.8464,  0.0134, -0.7864,  0.3056, -0.8285,
           -0.2573, -0.2673, -0.8733, -0.7007,  0.3613, -0.0218,  0.6090,
            0.4130, -0.0483,  0.3893, -0.3184, -0.2923,  0.6199, -0.7247,
            0.2650,  0.6588,  0.0274, -0.2591, -0.8602,  0.2306,  0.8265,
            0.9662, -0.8706, -0.7617, -0.2992,  0.2833,  0.3742,  0.577

In [78]:
lossi = net.fit(max_iter=10000,  loader=trloader, lr=(2.5 * 1e-3));

9999 / 10000: 5.298220

In [79]:
print("Training set loss:", eval_model(net, trloader))
print("Validation set loss:", eval_model(net, valloader))

Training set loss: 5.1899220989723185
Validation set loss: 5.468868528988775


In [119]:
g2 = torch.Generator(device).manual_seed(2147483647 + 1)
hni = None

for _ in range(5):
    out = []
    context = [0] * block_size
    hni = None

    while True:
        context = torch.tensor(context).to(device).expand(1, 8)
        probs, hni = net(context, hn=hni)
        logits = torch.nn.functional.softmax(probs, dim=0)

        ix = torch.multinomial(logits, num_samples=1, generator=g2).item()

        context = list(context)[1:] + [ix]

        if ix == 0:
            break

        out.append(ix)

        # if len(out) > 5:
        #     break
        # print(i_s[ix])


    print("".join(i_s[i] + " " for i in out))

he says this 
for like 
the closed-end herbert <unk> rain who write to personally dr. chairman a december days N 
your tends 
de predictable barber 
